In [28]:
#The CIFAR-10 dataset consists 0f 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training
#images and 10000 test images.
#airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
#format/size of image is 32x32x3

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision #exclusive pytorch library for computer vision #it also contains datsets. like CIFAR-10, MNIST etc.
#it also has pretrained cnns which are trained on millions of images.
#it also has utils for image transformations. like #min-max scaling, normalize etc
#like image(0-255) -> scale(0-1) -> normalize(-1,+1)
from torchvision.datasets import CIFAR10

In [30]:
#datasets & dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
#this helps in applying transformations to images.
transform = transforms.Compose([
    transforms.ToTensor(), #jo bhi images hongi usko pytorch tensors mein convert kardega and automatically scales the images.
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])
# In CIFAR-10 we automatically have training and testing sets.
trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [31]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [32]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [33]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

# Building Cnn's

In [34]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            #nn.Conv2d(in_channels, no.of filters or out channels,....)
            nn.Conv2d(3,32,kernel_size=3,padding=1), #kernel_size=3 mtlb filter of 3x3.
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel_size, stride_value

            nn.Conv2d(32,64,kernel_size=3,padding=1), #kernel_size=3 mtlb filter of 3x3.
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel_size, stride_value

            nn.Conv2d(64,128,kernel_size=3,padding=1), #kernel_size=3 mtlb filter of 3x3.
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel_size, stride_value
            
        )
        #assuming humne flatten kardiya hai.
        self.fc_layers = nn.Sequential(
            #nn.linear(in_features, out_features)
            nn.Linear(4*4*128, 256), #256 random no. of neurons in hidden layer
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) #flattening step
        x = self.fc_layers(x)

        return x

In [35]:
model = CNN()

In [36]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [37]:
#Training the cnn

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) #FP
        loss = criterion(output, labels) #loss fnx
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3599203155778559
epoch=2/10 & loss=0.927321371519962
epoch=3/10 & loss=0.7456030262553174
epoch=4/10 & loss=0.6099704507442997
epoch=5/10 & loss=0.5030800430937801
epoch=6/10 & loss=0.4071150122548613
epoch=7/10 & loss=0.31996876512989975
epoch=8/10 & loss=0.24855191947515967
epoch=9/10 & loss=0.18732693514612783
epoch=10/10 & loss=0.14719226322305934


In [39]:
#Evaluate our Cnn

total = 0
correct = 0

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        max_val, predicted = torch.max(outputs,1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)


print("total_vals:", total)
print("correct_vals:", correct)
print("Accuracy_score:", correct/total)

#You can add validation step in training for better results and comaprison.

total_vals: 10000
correct_vals: 7476
Accuracy_score: 0.7476


In [40]:
#Evaluate our Cnn

total = 0
correct = 0

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        max_val, predicted = torch.max(outputs,1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)


print("total_vals:", total)
print("correct_vals:", correct)
print("Accuracy_score:", correct/total)

total_vals: 10000
correct_vals: 7476
Accuracy_score: 0.7476
